# 12 · Random Forest (Bagging)
> **Part:** Part 1 · Classical supervised learning  
> **Family:** `Supervised · Classification & Regression (ensemble, variance reduction)`

## Definition
A random forest trains many *deep* decision trees, each on a different **bootstrap sample** (rows drawn with replacement) and
each split considering only a **random subset of features**, then averages their votes. This is **bagging** plus feature randomness.

## Intuition
One deep tree has low bias but very high variance. Averaging many *different* high-variance trees cancels their individual
mistakes: the errors are partly independent, so the average is far steadier. Randomness decorrelates the trees, which
is what makes averaging effective (variance of the mean ≈ ρσ² + (1−ρ)σ²/B).


## The Math

```text
for b = 1..B:  draw bootstrap sample D_b (n rows with replacement), grow tree T_b (random m features per split)
    prediction:    classification = majority vote / mean of probabilities ;  regression = mean
    Out-of-bag (OOB) estimate: each row is left out of ≈ 36.8 % of the trees, so score it with only those trees -> a free validation score.
    Feature importance: mean impurity decrease, or permutation importance.
```


In [ ]:
# Setup, Imports & Styling
%matplotlib inline
import os
import pathlib
import sys
from IPython.display import Image, display

# Ensure parent folder is in path to import mlatlas
parent_dir = pathlib.Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlatlas import data, viz, cli

# Ensure interactive inline mode and configure Seaborn theme
plt.ion()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})


### Step 1: Noisy Multi-Class Landscape Inspection
We inspect a complex classification dataset where individual decision trees tend to overfit.


In [ ]:
X, y = data.moons(n=300, noise=0.3, seed=0)

plt.figure(figsize=(8, 4.5))
sns.scatterplot(x=X[:, 0], y=X[:, 1], hue=y, palette=["#4c78a8", "#e45756"], s=50, edgecolor="k")
plt.title("Noisy Moons Dataset: Forest Aggregation", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### Step 2: From-Scratch Algorithm Implementation
Below is the complete implementation of the model components in pure NumPy.


In [ ]:
# Main Algorithm & Training Routine
def main(gif, quick):
    Xtr, ytr = data.moons(300, .35, seed=1); Xte, yte = data.moons(300, .35, seed=2); n = len(ytr)
    M = 12 if quick else 60; rng = np.random.default_rng(0); xx, yy, G = viz.grid(Xtr); trees = []; oob = np.zeros((M, n), bool)
    for m in range(M):
        idx = rng.integers(0, n, n); oob[m, np.setdiff1d(np.arange(n), idx)] = True
        trees.append(Tree(12, task="clf", max_features=1, rng=rng, n_classes=2).fit(Xtr[idx], ytr[idx]))
    Pg = np.array([t.predict_proba(G)[:, 1] for t in trees]); Pte = np.array([t.predict_proba(Xte)[:, 1] for t in trees]); Ptr = np.array([t.predict_proba(Xtr)[:, 1] for t in trees])
    cte = np.cumsum(Pte, 0) / np.arange(1, M + 1)[:, None]; num, den = np.cumsum(Ptr * oob, 0), np.cumsum(oob, 0)
    acc_te = [((cte[m] > .5) == yte).mean() for m in range(M)]
    acc_oob = [(((num[m] / np.maximum(den[m], 1)) > .5) == ytr)[den[m] > 0].mean() for m in range(M)]
    single = np.mean([((Pte[m] > .5) == yte).mean() for m in range(M)]); print(f"  average single-tree test accuracy {single:.3f}  ->  forest of {M}: {acc_te[-1]:.3f}")
    fr = sorted({m for m in [1, 2, 3, 4, 6, 9, 14, 20, 30, 45, 60] if m <= M})
    fig, (a1, a2, a3) = viz.subplots(1, 3, w=14, h=4.3)

    def update(m):
        for a in (a1, a2, a3): a.clear()
        viz.show_map(a1, xx, yy, Pg[:m].mean(0), prob=True); a1.contour(xx, yy, Pg[:m].mean(0).reshape(xx.shape), levels=[.5], colors="k", linewidths=1.8); viz.scatter(a1, Xtr, ytr, s=12)
        a1.set(title=f"Forest of {m} tree(s): test acc {acc_te[m - 1]:.2f}", xlim=(xx.min(), xx.max()), ylim=(yy.min(), yy.max()))
        viz.show_map(a2, xx, yy, (Pg[m - 1] > .5).astype(int)); viz.scatter(a2, Xtr, ytr, s=12); a2.set(title=f"Tree #{m} alone (jagged, overfit)", xlim=(xx.min(), xx.max()), ylim=(yy.min(), yy.max()))
        a3.plot(range(1, m + 1), acc_te[:m], color=viz.PALETTE[3], label="test"); a3.plot(range(1, m + 1), acc_oob[:m], color=viz.PALETTE[2], label="out-of-bag")
        a3.axhline(single, color="gray", ls=":", label="avg single tree"); a3.set(title="Accuracy vs number of trees", xlabel="trees", xlim=(0, M), ylim=(.7, .95)); a3.legend(loc="lower right")
    viz.animate(fig, update, fr, gif, fps=2)


### Step 3: Single Tree vs. Ensemble Averaging
We compare the erratic, jagged boundary of a single deep tree with the smooth, robust decision boundary of a 100-tree Random Forest.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

single_tree = DecisionTreeClassifier(random_state=42).fit(X, y)
forest = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))

Z1 = single_tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax1.contourf(xx, yy, Z1, alpha=0.35, cmap="coolwarm")
ax1.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=25, edgecolor="k")
ax1.set_title(f"Single Decision Tree: Jagged Boundary (Acc: {single_tree.score(X, y):.2f})", fontsize=11, fontweight="bold")

Z2 = forest.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax2.contourf(xx, yy, Z2, alpha=0.35, cmap="coolwarm")
ax2.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=25, edgecolor="k")
ax2.set_title(f"Random Forest (100 Trees): Smooth Boundary (Acc: {forest.score(X, y):.2f})", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()


### Step 4: Industry Benchmark (Out-of-Bag Score Validation)
We verify Out-Of-Bag (OOB) score approximation without a separate validation set.


In [ ]:
rf_oob = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42).fit(X, y)
print(f"Random Forest OOB Generalization Score: {rf_oob.oob_score_:.4f}")


### Step 5: Visualizing Learning Dynamics

Left: the forest's averaged probability map as trees are added (1 → 60), smoothing from a jagged single-tree map into a gentle boundary.
Middle: the newest single tree on its own (jagged!). Right: test accuracy and OOB accuracy converge.


In [ ]:
# Display the learning animation inline
media_file = f"../outputs/12_random_forest.gif"
if os.path.exists(media_file):
    display(Image(filename=media_file))
elif os.path.exists(f"outputs/12_random_forest.gif"):
    display(Image(filename=f"outputs/12_random_forest.gif"))


## Practical tips

- Few hyper-parameters, hard to overfit by adding trees, strong default for tabular data, no scaling needed.
- Slower/larger than a single tree; less interpretable; poor at extrapolating beyond the training range.


## 🧠 Check Your Understanding

<details>
<summary><b>1. Why does averaging multiple uncorrelated trees reduce variance without increasing bias?</b></summary>
If each tree has variance $\sigma^2$ and the trees are uncorrelated, averaging $B$ trees reduces variance to $\sigma^2 / B$. Because each tree is identically distributed, the expected value (and thus bias) remains unchanged.
</details>

<details>
<summary><b>2. How does feature subsampling (mtry) decorrelate trees?</b></summary>
If one feature is overwhelmingly predictive, every bootstrap tree would split on it first, making the trees highly correlated. By restricting each split to a random subset of $\sqrt{d}$ features, other features get selected, decorrelating the ensemble.
</details>


## 🧪 Guided Experiments

- Set `max_features=None` (all features): trees become more correlated; does the forest improve as quickly?
- Compare with a single depth-12 tree's test accuracy.
